Computing VPD files from the bias corrected inputs using the functions defined in VPD_functions.py

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client(n_workers=7, threads_per_worker=1) 
#client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36609,Workers: 7
Dashboard: /proxy/8787/status,Total threads: 7
Started: Just now,Total memory: 251.19 GiB
Comm: tcp://127.0.0.1:39643,Total threads: 1
Dashboard: /proxy/42875/status,Memory: 35.88 GiB
Nanny: tcp://127.0.0.1:37117,


2025-10-01 08:28:01,309 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/CMCC-ESM2/ssp370/r1i1p1f1/CCAM-v2203-SN/v1-r1/day/ssp370_CMCC-ESM2_CCAM-v2203-SN_QME_gwl1.2_vpd.nc', lease_id='cd6fde5ef72648fb87868fe08059b8ad'. This can happen if the Lock or Semaphore timed out before.
2025-10-01 08:30:26,397 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/CMCC-ESM2/ssp370/r1i1p1f1/CCAM-v2203-SN/v1-r1/day/ssp370_CMCC-ESM2_CCAM-v2203-SN_QME_gwl1.2_vpd.nc', lease_id='44aebd648a524fdf9b33db594c846c93'. This can happen if the Lock or Semaphore timed out before.
2025-10-01 09:28:54,372 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/CMCC-ESM2/ssp370/r1i1p1f1/CCAM-v2203-SN/v1-r1/day/ssp370_CMCC-ESM2_CCAM-v2203-SN_QME_gwl1.5_vpd.nc', lease_

In [2]:
#import all the stuff
from netCDF4 import Dataset
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/vpd/VPD")
import VPD_functions as VPD
sys.path.append("/g/data/mn51/users/nb6195/project/gwls/")
import gwl

In [78]:
#Set parameters
CMIP='CMIP6'
#AGENCY = 'CSIRO' 
#RCM = 'CCAM-v2203-SN'
AGENCY = 'BOM' 
RCM = 'BARPA-R'

#GCM = 'ACCESS-CM2' ensemble = 'r4i1p1f1' #Done
#GCM = 'ACCESS-ESM1-5' ensemble = 'r6i1p1f1' #Done
#GCM = 'EC-Earth3' ensemble = 'r1i1p1f1' #Done
GCM = 'MPI-ESM1-2-HR' 
ensemble = 'r1i1p1f1' #BOM done, no CSIRO
#GCM = 'CESM2' ensemble = 'r11i1p1f1' #Done
#GCM = 'CMCC-ESM2' ensemble = 'r1i1p1f1' #Done
#GCM = 'NorESM2-MM' ensemble = 'r1i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' ensemble = 'r1i1p1f2' #CSIRO Done, no BOM

#pathway = 'ssp126'
pathway = 'ssp370'

#bc_method = 'MRNBC'
bc_method = 'QME'

ddir = f"/g/data/kj66/CORDEX/output-{CMIP}/bias-adjusted-output/AUST-05i/{AGENCY}/{GCM}"
output_dir = '/g/data/ia39/ncra/bushfire/vpd/'
output_dir_mm = '/g/data/ia39/ncra/bushfire/vpd/monthly_mean/'

In [79]:
var1 = 'tasmaxAdjust'
var2 = 'hursminAdjust'

In [80]:
#read in files

#tasmax
infiles1a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var1}/v20241216/{var1}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
infiles1b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var1}/v20241216/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
tasmax_master_ds = xr.open_mfdataset(infiles1a + infiles1b)

#hursmin
infiles2a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
infiles2b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1-ACS-{bc_method}-BARRAR2-1980-2022/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1-ACS-{bc_method}-BARRAR2-1980-2022_day_*.nc')
hursmin_master_ds = xr.open_mfdataset(infiles2a + infiles2b)

In [81]:
#Extract time period corresponding to the chosen GWL for tasmax and rh
chosen_gwl = '1.2'

gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]

In [82]:
#Create the datasets for vpd and monthly mean vpd
gwl_vpd, monthly_mean_vpd = VPD.vpd_calc(gwl_rh, gwl_tasmax, chosen_gwl)

In [86]:
#print vpd to an external file
file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_' + bc_method + '_gwl' + chosen_gwl + '_vpd.nc'
output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

/g/data/ia39/ncra/bushfire/vpd/MPI-ESM1-2-HR/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_MPI-ESM1-2-HR_BARPA-R_QME_gwl1.2_vpd.nc


In [87]:
#print monthly mean ds to external file

file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + bc_method + '_' + pathway + '_' + ensemble + '.nc' 
output_file_location = output_dir_mm + file_name_mean
monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

/g/data/ia39/ncra/bushfire/vpd/monthly_mean/gwl1.2_monthly_mean_vpd_MPI-ESM1-2-HR_BARPA-R_QME_ssp370_r1i1p1f1.nc


In [ ]:
#for chosen_gwl in ['1.2', '1.5', '2.0', '3.0']:
#    gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
#    gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]

#    gwl_vpd, monthly_mean_vpd = VPD.vpd_calc(gwl_rh, gwl_tasmax, chosen_gwl)

#    file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_' + bc_method + '_gwl' + chosen_gwl + '_vpd.nc'
#    output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
#    gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
#    print(output_file_location)

#    file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + bc_method + '_' + pathway + '_' + ensemble + '.nc' 
#    output_file_location = output_dir_mm + file_name_mean
#    monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
#    print(output_file_location)

2025-09-10 14:05:54,033 - distributed.worker - ERROR - Compute Failed
Key:       ('mul-store-map-6334065ac0d073f0b8e5577d94a48bb5', 98, 0, 0)
State:     executing
Task:  <Task ('mul-store-map-6334065ac0d073f0b8e5577d94a48bb5', 98, 0, 0) store_chunk(...)>
Exception: "OSError(-101, 'NetCDF: HDF error')"
Traceback: '  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-24.12/lib/python3.11/site-packages/dask/array/core.py", line 4575, in store_chunk\n    return load_store_chunk(x, out, index, lock, return_stored, False)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-24.12/lib/python3.11/site-packages/dask/array/core.py", line 4557, in load_store_chunk\n    out[index] = x\n    ~~~^^^^^^^\n  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-24.12/lib/python3.11/site-packages/xarray/backends/netCDF4_.py", line 79, in __setitem__\n    data = self.get_array(needs_lock=False)\n           ^^^^^^^^^^^^^^